In [1]:
import requests
from dotenv import load_dotenv

load_dotenv()  # Loads variables from .env into the environment

True

In [6]:
"""
weather.py

Geocoding:       Nominatim (OpenStreetMap) — free, no API key required
Historical data: Open-Meteo Archive API    — free, no API key required

Usage:
    pip install requests
"""

import time

# ── Constants ────────────────────────────────────────────────────────────────

NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"

# Required by Nominatim's usage policy — identify your app
HEADERS = {"User-Agent": "my-weather-app/1.0 (your-email@example.com)"}


# ── Geocoding (Nominatim) ────────────────────────────────────────────────────


def get_coords_by_city(
    city_name: str,
    state: str = "",
    country_code: str = "",
) -> dict:
    """
    Get coordinates for a city name using the Nominatim (OpenStreetMap) API.

    Args:
        city_name:    Name of the city, e.g. "Chicago"
        state:        State name or code, e.g. "Illinois" or "IL" (optional)
        country_code: ISO 3166 country code, e.g. "US" (optional)

    Returns:
        Dict with 'name', 'lat', 'lon', and 'display_name'.
    """
    query = city_name
    if state:
        query += f", {state}"
    if country_code:
        query += f", {country_code}"

    params = {
        "q": query,
        "format": "json",
        "limit": 1,
        **({"countrycodes": country_code.lower()} if country_code else {}),
    }

    response = requests.get(NOMINATIM_URL, params=params, headers=HEADERS, timeout=10)
    response.raise_for_status()
    results = response.json()

    if not results:
        raise ValueError(f"No location found for: {query}")

    result = results[0]
    return {
        "name": result.get("display_name"),
        "lat": float(result["lat"]),
        "lon": float(result["lon"]),
    }


def get_coords_by_zip(
    zip_code: str,
    country_code: str = "US",
) -> dict:
    """
    Get coordinates for a ZIP/postal code using Nominatim.

    Args:
        zip_code:     ZIP or postal code, e.g. "10001"
        country_code: ISO 3166 country code (default: "US")

    Returns:
        Dict with 'name', 'lat', and 'lon'.
    """
    params = {
        "postalcode": zip_code,
        "countrycodes": country_code.lower(),
        "format": "json",
        "limit": 1,
    }

    response = requests.get(NOMINATIM_URL, params=params, headers=HEADERS, timeout=10)
    response.raise_for_status()
    results = response.json()

    if not results:
        raise ValueError(f"No location found for ZIP code: {zip_code}")

    result = results[0]
    return {
        "name": result.get("display_name"),
        "lat": float(result["lat"]),
        "lon": float(result["lon"]),
    }


# ── Historical Weather (Open-Meteo) ──────────────────────────────────────────


def get_historical_weather(
    lat: float,
    lon: float,
    date: str,
    temperature_unit: str = "fahrenheit",
    wind_speed_unit: str = "mph",
    precipitation_unit: str = "inch",
) -> dict:
    """
    Get historical weather for a specific day using the Open-Meteo Archive API.
    Free, no API key required. Data available from 1940 onwards.

    Args:
        lat:                Latitude
        lon:                Longitude
        date:               Date string in "YYYY-MM-DD" format, e.g. "2024-07-04"
        temperature_unit:   "celsius" or "fahrenheit" (default: "fahrenheit")
        wind_speed_unit:    "kmh", "mph", "ms", or "kn" (default: "mph")
        precipitation_unit: "mm" or "inch" (default: "inch")

    Returns:
        Dict with hourly and daily weather data for the requested day.
    """
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": date,
        "end_date": date,
        "hourly": "temperature_2m,precipitation,wind_speed_10m,relative_humidity_2m,weather_code",
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max",
        "temperature_unit": temperature_unit,
        "wind_speed_unit": wind_speed_unit,
        "precipitation_unit": precipitation_unit,
        "timezone": "auto",
    }

    response = requests.get(OPEN_METEO_URL, params=params, timeout=10)
    response.raise_for_status()
    return response.json()


# ── Convenience wrappers ─────────────────────────────────────────────────────


def get_weather_for_city(
    city_name: str,
    date: str,
    state: str = "",
    country_code: str = "",
    temperature_unit: str = "fahrenheit",
) -> dict:
    """
    Look up a city by name, then fetch its historical weather for a given date.

    Args:
        city_name:        Name of the city
        date:             Date in "YYYY-MM-DD" format
        state:            State name or code (optional)
        country_code:     ISO 3166 country code (optional)
        temperature_unit: "celsius" or "fahrenheit"

    Returns:
        Dict with 'location' and 'weather' keys.
    """
    location = get_coords_by_city(city_name, state, country_code)
    time.sleep(1)  # Nominatim usage policy: max 1 request/second
    weather = get_historical_weather(
        location["lat"], location["lon"], date, temperature_unit
    )
    return {"location": location, "weather": weather}


def get_weather_for_zip(
    zip_code: str,
    date: str,
    country_code: str = "US",
    temperature_unit: str = "fahrenheit",
) -> dict:
    """
    Look up a ZIP code, then fetch historical weather for a given date.

    Args:
        zip_code:         ZIP or postal code
        date:             Date in "YYYY-MM-DD" format
        country_code:     ISO 3166 country code (default: "US")
        temperature_unit: "celsius" or "fahrenheit"

    Returns:
        Dict with 'location' and 'weather' keys.
    """
    location = get_coords_by_zip(zip_code, country_code)
    time.sleep(1)  # Nominatim usage policy: max 1 request/second
    weather = get_historical_weather(
        location["lat"], location["lon"], date, temperature_unit
    )
    return {"location": location, "weather": weather}

In [5]:
# Example 1: by city name
print("--- By City ---")
result = get_weather_for_city(
    city_name="Chicago",
    state="Illinois",
    country_code="US",
    date="2024-07-04",
)
loc = result["location"]
daily = result["weather"]["daily"]
print(f"Location: {loc['name']}")
print("Date:     2024-07-04")
print(f"High:     {daily['temperature_2m_max'][0]}°F")
print(f"Low:      {daily['temperature_2m_min'][0]}°F")
print(f"Rain:     {daily['precipitation_sum'][0]} in")

print()

# Example 2: by ZIP code
print("--- By ZIP ---")
result = get_weather_for_zip(zip_code="10001", date="2024-07-04")
loc = result["location"]
daily = result["weather"]["daily"]
print(f"Location: {loc['name']}")
print("Date:     2024-07-04")
print(f"High:     {daily['temperature_2m_max'][0]}°F")
print(f"Low:      {daily['temperature_2m_min'][0]}°F")
print(f"Rain:     {daily['precipitation_sum'][0]} in")

--- By City ---


HTTPError: 403 Client Error: Forbidden for url: https://nominatim.openstreetmap.org/search?q=Chicago%2C+Illinois%2C+US&format=json&limit=1&countrycodes=us

In [7]:
get_coords_by_city("toronto")

HTTPError: 403 Client Error: Forbidden for url: https://nominatim.openstreetmap.org/search?q=toronto&format=json&limit=1